In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.spatial.distance import cdist

from shared.rlhc import random_latin_hypercube, space_filling_latin_hypercube
from shared.sample_funcs import peaks, appendix_one, branin
from shared.psi import psi
from shared.rbf import RBF, Parametric, Kriging

# 2 Constructing a Surrogate

## 2.3 Radial Basis Function Models

$$f(x) = \bm{w}^T \bm{\psi} = \sum_{i=1}^{n_c}w_i\psi(\left\lVert x-c^{(i)} \right\rVert)$$

In [ ]:
# def psi_gram_matrix(XA: np.ndarray, XB: np.ndarray, code: int = 1) -> np.ndarray:
    # original approach
    # n = X.shape[0]
    # PSI = np.zeros((n,n))

    # for i in range(n):
    #     for j in range(n):
    #         dist = np.linalg.norm(XA[i] - XB[j])
    #         PSI[i,j] = psi(dist,code)
    # return PSI

    # claude suggestion
    # D = cdist(XA, XB)
    # return psi(D, code)

In [ ]:
# 1D APPENDIX EXAMPLE
X_sample   = np.linspace(0,1,25).reshape(-1,1)
Y_sample   = appendix_one(X_sample)

D_sample   = cdist(X_sample, X_sample)
PSI_sample = psi(D_sample, code = 3)

W = np.linalg.solve(PSI_sample, Y_sample)

X   = np.linspace(0,1,100).reshape(-1,1)
D   = cdist(X, X_sample)
PSI = psi(D, code = 3)

func_latex = r'$f(x) = (6x - 2)^2 \sin(12x - 4)$'

fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(12, 5))

# --- RBF approximation ---
y = PSI @ W
ax1.plot(X, y)
ax1.set_title('Radial Basis Function Approximation')
ax1.set_xlabel(r'$x$')
ax1.set_ylabel(r'$\hat{f}(x)$')

# --- Actual function ---
y = appendix_one(X)
ax2.plot(X, y)
ax2.set_title('Actual Function')
ax2.set_xlabel(r'$x$')

fig.suptitle(func_latex, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
func_latex = (r'$f(x,y) = 3(1-x)^2 e^{-x^2-(y+1)^2} '
              r'- 10\left(\frac{x}{5} - x^3 - y^5\right) e^{-x^2-y^2} '
              r'- \frac{1}{3} e^{-(x+1)^2-y^2}$')

# --- Evaluate both surfaces on a grid ---
x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)
matr = np.hstack((X.reshape((-1, 1)), Y.reshape((-1, 1))))

Z_true = peaks(matr).reshape((200,200))

# --- Plot side by side ---
sample_sizes = [5,10,25,50,100]
n_plots = len(sample_sizes) + 1 

fig = plt.figure(figsize=(8, 6*n_plots))

ax = fig.add_subplot(n_plots, 1, 1, projection='3d')
surf2 = ax.plot_surface(X, Y, Z_true, cmap='viridis', linewidth=0, antialiased=True)
ax.set_title('Actual Function')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_zlabel(r'$f(x,y)$')
fig.colorbar(surf2, ax=ax, shrink=0.5, aspect=10)

for i, size in enumerate(sample_sizes):

    # --- Fit the RBF surrogate ---
    X_sample   = space_filling_latin_hypercube(size, 2, [-3]*2, [3]*2)
    D_sample   = cdist(X_sample, X_sample)
    PSI_sample = psi(D_sample, code = 3)
    Z_sample   = peaks(X_sample)
    W = np.linalg.solve(PSI_sample, Z_sample)

    D    = cdist(matr, X_sample)
    PSI  = psi(D, code = 3)
    Z_rbf  = (PSI @ W).reshape((200, 200))

    ax2 = fig.add_subplot(n_plots, 1, i+2, projection='3d')
    surf1 = ax2.plot_surface(X, Y, Z_rbf, cmap='viridis', linewidth=0, antialiased=True)
    ax2.set_title(f'Radial Basis Function Approximation; sample_count: {size}')
    ax2.set_xlabel(r'$x$')
    ax2.set_ylabel(r'$y$')
    ax2.set_zlabel(r'$\hat{f}(x,y)$')
    fig.colorbar(surf1, ax=ax2, shrink=0.5, aspect=10)

fig.suptitle(func_latex, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
x = np.linspace(-3,3,100)
y = np.logspace(-3,3,100)

plt.plot(x, y)

plt.show()

In [ ]:
func_latex = (r'$f(x,y) = 3(1-x)^2 e^{-x^2-(y+1)^2} '
              r'- 10\left(\frac{x}{5} - x^3 - y^5\right) e^{-x^2-y^2} '
              r'- \frac{1}{3} e^{-(x+1)^2-y^2}$')

# --- Evaluate both surfaces on a grid ---
x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)
matr = np.hstack((X.reshape((-1, 1)), Y.reshape((-1, 1))))

Z_true = peaks(matr).reshape((200,200))

# --- Plot side by side ---
sample_sizes = [5,10,25,50,100]
n_plots = len(sample_sizes) + 1 

fig = plt.figure(figsize=(8, 6*n_plots), dpi=100)

ax = fig.add_subplot(n_plots, 1, 1, projection='3d')
surf2 = ax.plot_surface(X, Y, Z_true, cmap='viridis', linewidth=0, antialiased=True)
ax.set_title('Actual Function')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_zlabel(r'$f(x,y)$')
fig.colorbar(surf2, ax=ax, shrink=0.5, aspect=10)

code = 6
for i, size in enumerate(sample_sizes):

    # --- Fit the RBF surrogate ---
    k = 2
    X_sample = space_filling_latin_hypercube(size, k, [-3]*k, [3]*k)

    rbf = Parametric(X_sample, peaks, code=code)
    sigma      = rbf.optimizer()
    PSI_sample = rbf.get_psi_matrix(X_sample, X_sample, sigma)
    Z_sample   = peaks(X_sample)
    W          = np.linalg.solve(PSI_sample, Z_sample)

    D    = cdist(matr, X_sample)
    PSI  = psi(D, code, sigma)
    Z_rbf  = (PSI @ W).reshape((200, 200))

    ax2 = fig.add_subplot(n_plots, 1, i+2, projection='3d')
    surf1 = ax2.plot_surface(X, Y, Z_rbf, cmap='viridis', linewidth=0, antialiased=True)
    ax2.set_title(f'Radial Basis Function Approximation; basis function: multi-quadratic; size: {size}')
    ax2.set_xlabel(r'$x$')
    ax2.set_ylabel(r'$y$')
    ax2.set_zlabel(r'$\hat{f}(x,y)$')
    fig.colorbar(surf1, ax=ax2, shrink=0.5, aspect=10)

fig.suptitle(func_latex, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
Nside = 200
x = np.linspace(-5, 10, Nside)
y = np.linspace(0, 15, Nside)
X, Y = np.meshgrid(x, y)
matr = np.hstack((X.reshape((-1, 1)), Y.reshape((-1, 1))))
Z_true = branin(matr).reshape((Nside, Nside))

k = 2
n = 50
X_sample = space_filling_latin_hypercube(n, k, [-5, 0], [10, 15])
kriging = Kriging(X_sample, branin)

Z = np.ones([Nside * Nside])
for i in range(matr.shape[0]):
    Z[i] = kriging.evaluate(matr[i, :])
Z = Z.reshape((Nside, Nside))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

from matplotlib.colors import Normalize

vmin, vmax = Z_true.min(), Z_true.max()
levels = np.linspace(vmin, vmax, 21)
norm = Normalize(vmin=vmin, vmax=vmax)
# --- actual function ---
func_latex = (    
    r'$f(x,y) = \left(y - \frac{5.1}{4\pi^2}x^2 '
    r'+ \frac{5}{\pi}x - 6\right)^2 '
    r'+ 10\left(1 - \frac{1}{8\pi}\right)\cos(x) + 10$'
)
fill1 = ax1.contourf(X, Y, Z_true, levels=levels, norm=norm, cmap='Greys')
ax1.contour(X, Y, Z_true, levels=levels, colors='black', linewidths=0.8)
ax1.set_title(func_latex)
ax1.set_xlabel(r'$x$')
ax1.set_ylabel(r'$y$')
ax1.set_aspect('equal')
fig.colorbar(fill1, ax=ax1, shrink=0.5, aspect=10)

# --- kriging surrogate ---
fill2 = ax2.contourf(X, Y, Z, levels=levels, norm=norm, cmap='Greys')
ax2.contour(X, Y, Z, levels=levels, colors='black', linewidths=0.8)
ax2.scatter(kriging.X_sample[:, 0], kriging.X_sample[:, 1],
            color='black', marker='^', s=7.5)
ax2.set_title(rf'$\hat{{f}}(x,y)$ ($n={len(kriging.X_sample)}$)')
ax2.set_xlabel(r'$x$')
ax2.set_ylabel(r'$y$')
ax2.set_aspect('equal')
fig.colorbar(fill2, ax=ax2, shrink=0.5, aspect=10)

In [ ]:
func_latex = (r'$f(x,y) = 3(1-x)^2 e^{-x^2-(y+1)^2} '
              r'- 10\left(\frac{x}{5} - x^3 - y^5\right) e^{-x^2-y^2} '
              r'- \frac{1}{3} e^{-(x+1)^2-y^2}$')

# --- Evaluate both surfaces on a grid ---
x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)
matr = np.hstack((X.reshape((-1, 1)), Y.reshape((-1, 1))))

Z_true = peaks(matr).reshape((200,200))

# --- Plot side by side ---
sample_sizes = [5,10,25,50,100]
n_plots = len(sample_sizes) + 1 

fig = plt.figure(figsize=(8, 6*n_plots))

ax = fig.add_subplot(n_plots, 1, 1, projection='3d')
surf2 = ax.plot_surface(X, Y, Z_true, cmap='viridis', linewidth=0, antialiased=True)
ax.set_title('Actual Function')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y$')
ax.set_zlabel(r'$f(x,y)$')
fig.colorbar(surf2, ax=ax, shrink=0.5, aspect=10)

Nside = 200
for i, size in enumerate(sample_sizes):

    # --- Fit the RBF surrogate ---
    X = random_latin_hypercube(size, 2, [-3]*2, [3]*2)
    X_sample = space_filling_latin_hypercube(X)
    
    kriging = Kriging(X_sample, peaks)
    Z_krig = np.ones([Nside*Nside])

    for j in range(matr.shape[0]):
        Z_krig[j] = kriging.evaluate(matr[j,:])
        
    Z_krig = Z_krig.reshape((Nside, Nside))

    ax2 = fig.add_subplot(n_plots, 1, i+2, projection='3d')
    surf1 = ax2.plot_surface(X, Y, Z_krig, cmap='viridis', linewidth=0, antialiased=True)
    ax2.set_title(f'Kriging Approximation; size: {size}')
    ax2.set_xlabel(r'$x$')
    ax2.set_ylabel(r'$y$')
    ax2.set_zlabel(r'$\hat{f}(x,y)$')
    fig.colorbar(surf1, ax=ax2, shrink=0.5, aspect=10)

fig.suptitle(func_latex, fontsize=12)
plt.tight_layout()
plt.show()

In [8]:
import numpy as np
import math
from scipy.spatial.distance import pdist

def random_latin_hypercube(
        n:int, 
        k:int,
        lbs: list = [],
        ubs: list = []
    ) -> np.ndarray:
    """
    Parameters
    ----------
    n : int
        Number of sample points. Also the number of bins per dimension,
        since the plan places one point per bin.
    k : int
        Number of dimensions (design variables).
    lbs : list, optional
        List of lower bound of the domain in every dimension (default 0).
    ubs : list, optional
        List of upper bound of the domain in every dimension (default 1).
        Must satisfy ub > lb.
    
    Returns
    -------
    np.ndarray
        An (n, k) array of sample points. Each column is a permutation of
        the same n bin centres, mapped to [lb, ub].
    
    Raises
    ------
    ValueError
        if ubs[i] <= lbs[i]
        
        if not len(ubs) == len(lbs) == k
    """
    if ubs == [] or lbs == []:
        lbs = [0]*k
        ubs = [1]*k

    if not len(ubs) == len(lbs) == k:
        raise ValueError("Upper and Lower bounds do not match eachother or k")

    for ub, lb in zip(ubs, lbs):
        if ub <= lb:
            raise ValueError("upper bound must be greater than lower bound")
        
    rng = np.random.default_rng()

    X = np.zeros((n,k))
    for i in range(k):
        X[:,i] = rng.permutation(n)
    
    X_scaled = np.zeros((n,k))
    for i, (lb, ub) in enumerate(zip(lbs, ubs)):
        X[:,i] = (X[:,i] + 0.5)/n
        w = ub - lb

        X_scaled[:,i] = lb + w*X[:,i]

    return X_scaled

def dist(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # # My original approach:
    # distances = []
    # for i in range(len(X)-1):
    #     for j in range(1,len(X)):
    #         distances.append(np.linalg.norm(X[i,:] - X[j,:]))
    
    # # round because normalized to [0,1]
    # return np.unique(distances, return_counts=True) # returns distinct_d, J

    # claude suggested using pdists from scipy instead:
    sq_dists = pdist(X, metric='sqeuclidean')
    return np.unique(sq_dists, return_counts=True) # returns distinct_d, J


def mm_phiq(X,q=2):
    d,J = dist(X)

    # morris mitchell sampling plan quality criterion
    return sum(((J/(d**q))**(1/q)))


def perturb(X: np.ndarray, pert_n=1):
    n = X.shape[0]
    k = X.shape[1]

    for _ in range(pert_n):
        col = int(np.floor(np.random.random()*k))
        
        el1 = el2 = 1
        while el1==el2:
            el1 = int(np.floor(np.random.random()*n))
            el2 = int(np.floor(np.random.random()*n))

        buffer = X[el1,col]
        X[el1,col] = X[el2,col]
        X[el2,col] = buffer

    return X


def best_lhc_by_q(X_start, pop, iter, q = 2):
    X_best = X_start; 
    phi_best = mm_phiq(X_start, q)

    n = X_best.shape[0]

    leveloff = math.floor(0.85*iter)
    for it in range(1,iter+1):
        if it < leveloff:
            mutations = round(1+(0.5*n-1)*(leveloff-it)/(leveloff-1))
        else:
            mutations = 1
    
        X_improved   = X_best
        phi_improved = phi_best
        
        for _ in range(pop):
            X_try = perturb(X_best, mutations)
            phi_try = mm_phiq(X_try, q)
            
            if phi_try < phi_improved:
                X_improved = X_try
                phi_improved = phi_try
        
        if phi_improved < phi_best:
            X_best = X_improved
            phi_best = phi_improved

    return X_best, phi_best


def space_filling_latin_hypercube(
        X_start: np.ndarray,
        pop:int = 25,
        iter:int = 25
    ) -> dict:
    """
    Generate a random Latin hypercube sampling plan in k dimensions.

    Each dimension is partitioned into n equal-width bins, and the plan
    places exactly one point per bin in every dimension (the Latin
    hypercube property), giving more uniform coverage of the domain than
    independent uniform sampling. Points are positioned at bin centres and
    then scaled from the unit hypercube [0, 1]^k onto [lb, ub]^k.

    Parameters
    ----------
    X_start : np.ndarray
        A random latin hypercube generated via random_latin_hypercube
    pop : int, optional
        Controls the number of perturbations done to the random latin hypercube
    iter : int, optional
        Controls the number of mutations done during a given perturbation

    Increasing pop and iter will improve the sampling plan but increase time
    to compute.
    
    Returns
    -------
    np.ndarray
        An (n, k) array of sample points. Each column is a permutation of
        the same n bin centres.
    """
    qs = [1,2,5,10,20,50,100]

    X_tracker = {
        "q": "",
        "phi": math.inf,
        "X": ""
    }

    for q in qs:
        X_best, phi_best = best_lhc_by_q(X_start, pop, iter, q)

        if phi_best < X_tracker["phi"]:
            X_tracker["q"]   = q
            X_tracker["phi"] = phi_best
            X_tracker["X"]   = X_best
    
    return X_tracker["X"], X_tracker["q"]


In [10]:
from scipy.spatial.distance import pdist
import math


n = 25
k = 2
rlhc = random_latin_hypercube(n,k)
X_c, q = space_filling_latin_hypercube(rlhc)

def dist(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # # My original approach:
    # distances = []
    # for i in range(len(X)-1):
    #     for j in range(1,len(X)):
    #         distances.append(np.linalg.norm(X[i,:] - X[j,:]))
    
    # # round because normalized to [0,1]
    # return np.unique(distances, return_counts=True) # returns distinct_d, J

    # claude suggested using pdists from scipy instead:
    sq_dists = pdist(X, metric='sqeuclidean')
    return np.unique(sq_dists, return_counts=True) # returns distinct_d, J


def mm_phiq(X,q=2):
    d,J = dist(X)

    # morris mitchell sampling plan quality criterion
    return sum(((J/(d**q))**(1/q)))


def find_subset(cube: np.ndarray, n: int, q: int = 2, iter: int = 5) -> np.ndarray:
    rng = np.random.default_rng()

    X_tracker = {
        "phi": math.inf,
        "X": ""
    }

    for _ in range(iter):
        # restart to avoid local minima
        X_best   = rng.choice(cube, size=n, replace=False)
        phi_best = mm_phiq(X_e, q)

        if phi_best < X_tracker["phi"]:
            X_tracker["phi"] = phi_best
            X_tracker["X"]   = X_best

        for i in range(n):
            X_test = X_tracker["X"]
            print(X_test)
            for point in cube:
                X_test[i] = point
                phi_best  = mm_phiq(X_e, q)

                if phi_best < X_tracker["phi"]:
                    X_tracker["phi"] = phi_best
                    X_tracker["X"]   = X_test
        

    return X_tracker["X"]
    

X_e = find_subset(X_c, 10, q)
print(X_e)

/tmp/ipykernel_66505/3315952401.py:29: RuntimeWarning: divide by zero encountered in divide
  return sum(((J/(d**q))**(1/q)))


TypeError: 'str' object does not support item assignment